# Create Pipeline Package for segmentation with GPU

In this notebook, in contrast to notebook [30-CreatePipeline](./30-CreatePipeline.ipynb), the main goal is to create a GPU-enabled Pipeline and keep the GPU variants of PyTorch together with their related transitive dependencies for execution on AI Inference Server.

This GPU flow requires an AI Inference Server version with GPU support (2.9.0 or newer).

## Creating a PythonComponent wrapper

In this step, we create a `PythonComponent` that defines the entrypoint of the step we explained in notebook [20-CreateInferenceWrapper](./20-CreateInferenceWrapper.ipynb). The script itself is executed by AI Inference Server. We set `gpu_enabled=True` so the GPU-oriented dependency setup is preserved.\
To do so, we need to 
- collect our file resources,
- define component inputs and outputs,
- define the required Python environment.

The resources contain our Ultralytics YOLO model, the entrypoint python script [segmentation.py](../src/segmentation.py), and an additional helper script [imageset.py](../src/imageset.py). In order to make the Ultralytics python package work, we prepared a [requirements-gpu.txt](../src/requirements-gpu.txt) ([requirements-cpu.txt](../src/requirements-cpu.txt) for the CPU version), as explained in the [previous notebook](./20-CreateInferenceWrapper.ipynb). In this GPU flow, `requirements-gpu.txt` is intended to contain the related dependency set, including transitive dependencies needed at runtime.

In [ ]:
from simaticai import deployment

model_name = "yolo11n-seg.pt"

# component = deployment.PythonComponent("segmentation", python_version="3.12")
component = deployment.PythonComponent("segmentation", python_version="3.12", gpu_enabled=True)

component.add_resources("../src/", "segmentation.py")
component.add_resources("../models/", model_name)
component.set_entrypoint("segmentation.py")
# component.set_requirements("../src/requirements-cpu.txt", no_deps=True)
component.set_requirements("../src/requirements-gpu.txt", no_deps=True)

component.add_input("vision_payload", "ImageSet")

component.add_output("result_image_set", "ImageSet")
component.add_output("iuid", "String")
component.add_output("areas", "String")

Note that when we create the Python component, we set `gpu_enabled` true, indicating that we want to keep the GPU-enabled `torch` package. Otherwise, AI SDK replaces it with a CPU variant of `torch`.

The `imageset.py` file should have been created by notebook 20. In case it did not run, we create this file again here.

In [ ]:
from simaticai.common.resources import copy_resource_to
copy_resource_to('ImageSet', '../src')

Now we can add it as a resource to our component.

In [ ]:
component.add_resources("../src/", "imageset.py")
component

## Creating a Pipeline from the component

Now we can use the component to create a Pipeline configuration. The Pipeline requires a list of components - which is a single component in this example - and a name. Providing a description is optional.

In [ ]:
pipeline = deployment.Pipeline.from_components([component], name="Segmentation on GPU", version="1", desc="Segmentation tutorial with Ultralytics YOLO on GPU.")
pipeline

## Build the Pipeline Package

This step creates the proper content in the defined target folder `packages` and creates the edge Pipeline Package as a zip file. The export method first validates the Pipeline and raises an error if it finds any problems.

In [ ]:
edge_package_path = pipeline.export('../packages')
edge_package_path

## Test the Pipeline Package locally

We suggest to test the Pipeline Package on your computer before deploying it to the AI Inference Server. It is possible to do so using notebook [41-TestPipelineLocallyWithGPU](41-TestPipelineLocallyWithGPU.ipynb).